# Windowed UID Experiment

Active/Passive alternations — windowed token-level UID analysis.

**Runtime:** Change to A100 via *Runtime → Change runtime type → A100* for best performance.

---

## Cell 1 — GPU check

In [ ]:
import torch
print('GPU:  ', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — switch to A100 runtime')
print('CUDA: ', torch.version.cuda)
print('bf16: ', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

## Cell 2 — Install uv + clone repo

In [ ]:
%%bash
pip install uv -q
# Replace YOUR_USERNAME with your GitHub username
git clone https://github.com/YOUR_USERNAME/active-passive-alternations.git
cd active-passive-alternations && git checkout colab-monorepo
echo 'Done'

## Cell 3 — Install dependencies

In [ ]:
%%bash
cd active-passive-alternations
uv sync
echo 'Done'

## Cell 4 — Mount / upload data

Choose **Option A** (Google Drive) or **Option B** (direct upload).

In [ ]:
import os

# ── Option A: Google Drive ──────────────────────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/YOUR_DATA_PATH'   # <-- adjust

# ── Option B: Upload .conllu files ─────────────────────────────────────────
from google.colab import files
os.makedirs('/content/active-passive-alternations/data', exist_ok=True)
uploaded = files.upload()   # select your .conllu files
import shutil
for fn in uploaded:
    shutil.move(fn, f'/content/active-passive-alternations/data/{fn}')
DATA_DIR = '/content/active-passive-alternations/data'

print(f'DATA_DIR = {DATA_DIR}')
print(f'Files: {os.listdir(DATA_DIR)}')

## Cell 5 — Run windowed UID

Adjust `--limit_docs` and `--limit_sents_per_doc` for a quick smoke test vs. full run.

In [ ]:
%%bash
cd /content/active-passive-alternations
uv run python scripts/run_windowed_uid.py "$DATA_DIR" gpt2 \
    --context document \
    --uid_unit token \
    --uid_level "(-10,+10)" \
    --generate_counterfactual \
    --fast --batch_size 128 \
    --output_dir outputs \
    --output_name window_uid_tok10.csv \
    --limit_docs 50 \
    --limit_sents_per_doc 12

## Cell 6 — Check coverage

In [ ]:
%%bash
cd /content/active-passive-alternations
uv run python scripts/check_window_coverage.py \
    --csv outputs/window_uid_tok10.csv

## Cell 7 — Sweep windows (optional)

In [ ]:
%%bash
cd /content/active-passive-alternations
uv run python scripts/sweep_uid_windows.py "$DATA_DIR" gpt2 \
    --windows "(-0,+0)" "(-5,+5)" "(-10,+10)" "(-20,+20)" \
    --context document \
    --generate_counterfactual \
    --fast --batch_size 128 \
    --output_dir outputs \
    --output_name sweep_results.csv \
    --limit_docs 50 --limit_sents_per_doc 12

## Cell 8 — Preview results

In [ ]:
import pandas as pd
df = pd.read_csv('/content/active-passive-alternations/outputs/window_uid_tok10.csv')
print(f'Rows: {len(df)}')
print(df.head())

## Cell 9 — Download results

In [ ]:
from google.colab import files
files.download('/content/active-passive-alternations/outputs/window_uid_tok10.csv')
# Uncomment to also download sweep:
# files.download('/content/active-passive-alternations/outputs/sweep_results.csv')